In [1]:
import pandas as pd
from tools import hash_danych_bezpieczny

INTERIM_PATH = "dane/interim/"

In [2]:
df_fi_hard = pd.read_parquet(f"{INTERIM_PATH}fact_inka_hard.parquet")
print(f"Wczytano: {df_fi_hard.shape}")

Wczytano: (4106901, 30)


#### hash: **fact_inka_hard**  
4b6ce69cbeb2b5a1c6cd820a8857b4aacc16c126807fc5409b946caa8a68ed73

In [3]:
hash_wejsciowy = [
    ('fact_inka_hard', ['TowId', 'Data'])
]

for nazwa, kolumny in hash_wejsciowy:
    h = hash_danych_bezpieczny(f"{INTERIM_PATH}{nazwa}.parquet", kolumny_sortowania=kolumny)
    print(f"Hash {nazwa}: {h}")

Hash fact_inka_hard: 4b6ce69cbeb2b5a1c6cd820a8857b4aacc16c126807fc5409b946caa8a68ed73


In [4]:
# SKU bez żadnej sprzedaży
sku_ze_sprzedaza = set(df_fi_hard[df_fi_hard['TypDok'] == 21]['TowId'].unique())
sku_wszystkie = set(df_fi_hard['TowId'].unique())
sku_bez_sprzedazy = sku_wszystkie - sku_ze_sprzedaza

print(f"SKU bez żadnej sprzedaży: {len(sku_bez_sprzedazy)}")

# Filtrowanie
df_fi_soft = df_fi_hard[~df_fi_hard['TowId'].isin(sku_bez_sprzedazy)].copy()

print(f"Przed: {len(df_fi_hard):,} rekordów, {df_fi_hard['TowId'].nunique():,} SKU")
print(f"Po:    {len(df_fi_soft):,} rekordów, {df_fi_soft['TowId'].nunique():,} SKU")

SKU bez żadnej sprzedaży: 7211
Przed: 4,106,901 rekordów, 19,692 SKU
Po:    4,057,332 rekordów, 12,481 SKU


In [5]:
df_fi_soft.to_parquet(f"{INTERIM_PATH}fact_inka_soft.parquet", 
                       engine='pyarrow', index=False, compression='zstd')

print(f"Zapisano: {df_fi_soft.shape}")

Zapisano: (4057332, 30)


In [6]:
hash_wyjsciowy = [
    ('fact_inka_soft', ['TowId', 'Data'])
]

for nazwa, kolumny in hash_wyjsciowy:
    h = hash_danych_bezpieczny(f"{INTERIM_PATH}{nazwa}.parquet", kolumny_sortowania=kolumny)
    print(f"Hash {nazwa}: {h}")

Hash fact_inka_soft: 87ff0e55ac94368b71e9e36a33c43e321c2bfaeeaffae189daa9d0d8996cff10


#### Do weryfikacji hash końcowy: 87ff0e55ac94368b71e9e36a33c43e321c2bfaeeaffae189daa9d0d8996cff10